# INDICATION Detection — Rule-Out Context

INDICATION is the most clinically important category that medspaCy misses. It encodes the 
epistemic state *?(X)*: the clinician does not yet know whether X is present — and is actively 
trying to determine it. This is categorically different from negation.

In [1]:
import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])

## Why INDICATION ≠ Negation

"Rule out PE" means: *we do not yet know whether PE is present; imaging was ordered to answer that question.*
A negative report answer is DEFINITE_NEGATED_EXISTENCE. The indication sentence itself is neither — 
it is a statement of epistemic incompleteness.

Modal reading: INDICATION encodes ¬K(PE) ∧ ¬K(¬PE) — the clinician knows neither the positive 
nor the negative. Treating it as negation introduces a false finding of absence.

In [2]:
# Classic rule-out phrasings — medspaCy classifies these as is_negated=True
ruleout_sentences = [
    "Rule out PE.",
    "R/o pulmonary embolism.",
    "Evaluate for PE.",
]

print("Classic rule-out phrasings")
print("-" * 60)
for sent in ruleout_sentences:
    doc = nlp(sent)
    for ent in doc.ents:
        print(f"  {sent!r}")
        print(f"    medspaCy is_negated : {ent._.is_negated}")
        print(f"    cwyde category     : {ent._.cwyde_assertion_category.value}")
        print()

2026-04-28 22:32:30.463 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'Rule' marked as sentence start (span begin)


2026-04-28 22:32:30.463 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] Token/tag mapping: [(Rule, True), (out, False), (PE, False), (., False)]


2026-04-28 22:32:30.528 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'R' marked as sentence start (span begin)


2026-04-28 22:32:30.529 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] Token/tag mapping: [(R, True), (/, False), (o, False), (pulmonary, False), (embolism, False), (., False)]


2026-04-28 22:32:30.582 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'Evaluate' marked as sentence start (span begin)


2026-04-28 22:32:30.583 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] Token/tag mapping: [(Evaluate, True), (for, False), (PE, False), (., False)]


Classic rule-out phrasings
------------------------------------------------------------
  'Rule out PE.'
    medspaCy is_negated : False
    cwyde category     : INDICATION

  'R/o pulmonary embolism.'
    medspaCy is_negated : False
    cwyde category     : INDICATION

  'Evaluate for PE.'
    medspaCy is_negated : False
    cwyde category     : INDICATION



In [3]:
# Backfill patterns: concern/question-of/worrisome → INDICATION without "rule out"
backfill_sentences = [
    "Concern for PE.",
    "Question of pulmonary embolism.",
    "Worrisome for PE.",
]

print("Backfill INDICATION patterns (no explicit 'rule out')")
print("-" * 60)
for sent in backfill_sentences:
    doc = nlp(sent)
    for ent in doc.ents:
        print(f"  {sent!r}")
        print(f"    medspaCy is_negated : {ent._.is_negated}")
        print(f"    cwyde category     : {ent._.cwyde_assertion_category.value}")
        print()

2026-04-28 22:32:30.644 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] [doc 0] Token 0 'Concern' marked as sentence start (span begin)


2026-04-28 22:32:30.644 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=3] Token/tag mapping: [(Concern, True), (for, False), (PE, False), (., False)]


Backfill INDICATION patterns (no explicit 'rule out')
------------------------------------------------------------


2026-04-28 22:32:30.676 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] [doc 0] Token 0 'Question' marked as sentence start (span begin)


2026-04-28 22:32:30.676 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] Token/tag mapping: [(Question, True), (of, False), (pulmonary, False), (embolism, False), (., False)]


2026-04-28 22:32:30.708 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] [doc 0] Token 0 'Worrisome' marked as sentence start (span begin)


2026-04-28 22:32:30.709 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=5] Token/tag mapping: [(Worrisome, True), (for, False), (PE, False), (., False)]


  'Concern for PE.'
    medspaCy is_negated : False
    cwyde category     : INDICATION

  'Question of pulmonary embolism.'
    medspaCy is_negated : False
    cwyde category     : INDICATION

  'Worrisome for PE.'
    medspaCy is_negated : False
    cwyde category     : INDICATION



In [4]:
# Contrast: genuine negation — absence is confirmed, not under investigation
doc = nlp("No PE.")
for ent in doc.ents:
    print("Contrast: genuine negation")
    print(f"  'No PE.'")
    print(f"    medspaCy is_negated : {ent._.is_negated}")
    print(f"    cwyde category     : {ent._.cwyde_assertion_category.value}")
    print("  → DEFINITE_NEGATED_EXISTENCE: absence confirmed, not under investigation")

2026-04-28 22:32:30.744 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] [doc 0] Token 0 'No' marked as sentence start (span begin)


2026-04-28 22:32:30.744 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=6] Token/tag mapping: [(No, True), (PE, False), (., False)]


Contrast: genuine negation
  'No PE.'
    medspaCy is_negated : False
    cwyde category     : DEFINITE_NEGATED_EXISTENCE
  → DEFINITE_NEGATED_EXISTENCE: absence confirmed, not under investigation
